In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
def Read_data(filepath):
    df = pd.read_csv(f'{filepath}.csv')
    # df = df[df['fuente'] != 3]
    # df = df[df['calidad'] != 3]
    return df

def rangos_edades(edad):
    if pd.isna(edad):
        return np.nan
    elif edad < 10: 
        return '0-9'
    elif edad < 15:
        return '10-14'
    elif edad < 20:
        return '15-19'
    elif edad < 25:
        return '20-24'
    elif edad < 30:
        return '25-29'
    elif edad < 35:
        return '30-34'
    elif edad < 40:
        return '35-39'
    elif edad < 45:
        return '40-44'
    elif edad >= 45:
        return '45+'
    
def safe_convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return None
    
def Modify_data(df, name_date):
    df = df.sort_values(by=['fecha'])
    # df.drop_duplicates(subset=['exp'], keep='last', inplace=True)

    if 'escolar' in df.columns:
        df['escolar'] = df['escolar'].apply(safe_convert_to_float)
    if 'edad' in df.columns:
        df['edad'] = df['edad'].apply(safe_convert_to_float)
        df['drogaim'] = df['drogaim'].apply(safe_convert_to_float)

    name_date = re.search(r'\\data\\(.*)_1', name_date).group(1)
    year = int(re.search(r'(\d{4})([AB])', name_date).group(1))
    semester = re.search(r'(\d{4})([AB])', name_date).group(2)
    month = 1 if semester == 'A' else 7
    df['Año'] = year
    df['Mes'] = month
    df['Semestre'] = np.where(df['Mes'].between(1, 6), 1, 2)

    cols = [
    'tab1av','alcoav','mar1av','mar2av','coc1av','coc2av','coc3av','inh1av','inh2av',
    'inh3av','inh4av','est1av','est2av','est3av','est4av','alu1av','alu2av','alu3av','dep1av','dep2av','dep3av','opi1av','opi2av','opi3av','osu1av','osu2av'
    ]
    for col in cols:
        df[col] = df[col].apply(safe_convert_to_float)
        df[col] = df[col].apply(lambda x: x if x in [1, 2] else np.nan)
        df[col] = df[col].replace({2:0})


    dict_estado = pd.read_csv(f'{os.getcwd()}\\data\\CentrosDeCostoEstado.csv')
    dict_estado = dict(zip(dict_estado['CENTRO'], dict_estado['ESTADO']))
    dict3 = {'CIUDAD DE MÉXICO':"CDMX", 'ESTADO DE MÉXICO': "EMEX", 'JALISCO':"JAL", 'SINALOA': 'SIN', 'BAJA CALIFORNIA':"BC", 'CHIHUAHUA': "CHH", 'GUANAJUATO': 'GTO', 'QUINTANA ROO':"QNTROO", 'COAHUILA':  "COAH", 'NUEVO LEÓN':  "NVL", 'MICHOACÁN': "MICH", 'GUERRERO': "GRO", 'COLIMA': "COL", 'BAJA CALIFORNIA SUR':"BCS", 'TAMAULIPAS': "TAM",
        'VERACRUZ': "VRC", 'SONORA': "SON", 'PUEBLA': "PBL", 'DURANGO': "DUR", 'AGUASCALIENTES':  "AGS", 'YUCATÁN':"YCT", 'HIDALGO':"HDO",'ZACATECAS':"ZAC", 'QUERÉTARO': "QRO", 'SAN LUIS POTOSÍ': "SLP" , 'OAXACA': "OAX", 'TABASCO': "TBS", 'CHIAPAS': "CHPS", 'MORELOS':"MOR", 'TLAXCALA': "TLX", 'CAMPECHE':"CAM", 'NAYARIT':"NAY"}
    centros = pd.read_csv(r'C:\Users\franc\Documents\OneDrive\Documentos\GitHub\Recod_Historico\CentrosDeCosto.csv')
    dict_centros = dict(zip(centros['CentroCostoClave'], centros['CentroCostoId']))
    df['CentroCostoId'] = df['cec'].map(dict_centros)
    df['Estado'] = df['cec'].map(dict_estado).map(dict3)
    return df

def min_nonzero(*cols):
    return pd.DataFrame(cols).apply(lambda x: x[x > 0].min() if (x > 0).any() else 0, axis=0)

def Rango_UM(x):
    if x in range(1,4):
        return 1
    elif x == 4:
        return 0
    else: 
        return np.nan

def UltimoMes(df, df2):

    df2['TabacoUM'] = df['tab1uc']
    df2['AlcoholUM'] = df['alcouc']
    df2['MarihuanaUM'] = df['mar1uc']
    df2['HachisUM'] = df['mar2uc']
    df2['CocaínaUM'] = df['coc1uc']
    df2['CrackUM'] = df['coc2uc']
    df2['Otras Presentaciones (Basuco o pasta base, cocaina negra)UM'] = df['coc3uc']
    df2['Solventes y removedoresUM'] = df['inh1uc']
    df2['PegamentoUM'] = df['inh2uc']
    df2['Esmaltes y pinturasUM'] = df['inh3uc']
    df2['Otros (aire comprimido, gasolinas y combustibles)UM'] = df['inh4uc']
    df2['AnfetaminasUM'] = df['est1uc']
    df2['MetanfetaminasUM'] = df['est2uc']
    df2['MDMA(extasis) y metanfetaminas alucinogenas (DMT)UM'] = df['est3uc']
    df2['Otros (derivados anfetaminicos)UM'] = df['est4uc']
    df2['LSDUM'] = df['alu1uc']
    df2['Plantas alucinogenas y derivadosUM'] = df['alu2uc']
    df2['Otras (PCP, ketamina, excepto metanfetamina)UM'] = df['alu3uc']
    df2['BenzodiazepinasUM'] = df['dep1uc']
    df2['RohypnolUM'] = df['dep2uc']
    df2['Otras SustanciasUM'] = df['dep3uc']
    df2['HeroinaUM'] = df['opi1uc']
    df2['Opiaceos sinteticos (propoxifeno, nailbufina)UM'] = df['opi2uc']
    df2['Opio y opiodes (morfina, codeina)UM'] = df['opi3uc']
    df2['Con utilidad medica (Prozac, Paxil, Carbamazepina)UM'] = df['osu1uc']
    df2['Otras SustanciasUM'] = df['osu2uc']

    listcols = [
        'TabacoUM', 'AlcoholUM', 'MarihuanaUM', 'HachisUM', 'CocaínaUM', 'CrackUM',
        'Otras Presentaciones (Basuco o pasta base, cocaina negra)UM', 'Solventes y removedoresUM', 'PegamentoUM', 'Esmaltes y pinturasUM',
        'Otros (aire comprimido, gasolinas y combustibles)UM', 'AnfetaminasUM', 'MetanfetaminasUM', 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)UM',
        'Otros (derivados anfetaminicos)UM', 'LSDUM', 'Plantas alucinogenas y derivadosUM', 'Otras (PCP, ketamina, excepto metanfetamina)UM',
        'BenzodiazepinasUM', 'RohypnolUM', 'Otras SustanciasUM', 'HeroinaUM', 'Opiaceos sinteticos (propoxifeno, nailbufina)UM', 'Opio y opiodes (morfina, codeina)UM',
        'Con utilidad medica (Prozac, Paxil, Carbamazepina)UM', 'Otras SustanciasUM'
    ]
    for col in listcols:
        df2[col] = df2[col].apply(Rango_UM)
    return df2

def Edad_inicio(df, df2):

    df2['EdadInicioTabaco'] = df['tab1e']
    df2['EdadInicioAlcohol'] = df['alcoe']
    df2['EdadInicioMarihuana'] = df['mar1e']
    df2['EdadInicioHachis'] = df['mar2e']
    df2['EdadInicioCocaína'] = df['coc1e']
    df2['EdadInicioCrack'] = df['coc2e']
    df2['EdadInicioOtras Presentaciones (Basuco o pasta base, cocaina negra)'] = df['coc3e']
    df2['EdadInicioSolventes y removedores'] = df['inh1e']
    df2['EdadInicioPegamento'] = df['inh2e']
    df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
    df2['EdadInicioOtros (aire comprimido, gasolinas y combustibles)'] = df['inh4e']
    df2['EdadInicioAnfetaminas'] = df['est1e']
    df2['EdadInicioMetanfetaminas'] = df['est2e']
    df2['EdadInicioMDMA(extasis) y metanfetaminas alucinogenas (DMT)'] = df['est3e']
    df2['EdadInicioOtros (derivados anfetaminicos)'] = df['est4e']
    df2['EdadInicioLSD'] = df['alu1e']
    df2['EdadInicioPlantas alucinogenas y derivados'] = df['alu2e']
    df2['EdadInicioOtras (PCP, ketamina, excepto metanfetamina)'] = df['alu3e']
    df2['EdadInicioBenzodiazepinas'] = df['dep1e']
    df2['EdadInicioRohypnol'] = df['dep2e']
    df2['EdadInicioOtras Sustancias'] = df['dep3e']
    df2['EdadInicioHeroina'] = df['opi1e']
    df2['EdadInicioOpiaceos sinteticos (propoxifeno, nailbufina)'] = df['opi2e']
    df2['EdadInicioOpio y opiodes (morfina, codeina)'] = df['opi3e']
    df2['EdadInicioCon utilidad medica (Prozac, Paxil, Carbamazepina)'] = df['osu1e']
    df2['EdadInicioOtras Sustancias'] = df['osu2e']
    return df2

def DataEsp(df): 
    list_av_ileg = [
    'Marihuana', 'Hachis', 'Cocaína',
    'Crack', 'Otras Presentaciones (Basuco o pasta base, cocaina negra)', 'Otros (aire comprimido, gasolinas y combustibles)', 'Solventes y removedores', 'Pegamento',
    'Esmaltes y pinturas', 'Otros (derivados anfetaminicos)', 'Anfetaminas', 'Metanfetaminas',
    'MDMA(extasis) y metanfetaminas alucinogenas (DMT)', 'Otras Sustancias', 'Otras (PCP, ketamina, excepto metanfetamina)', 'Plantas alucinogenas y derivados', 'LSD',
    'Benzodiazepinas', 'Rohypnol', 'Opio y opiodes (morfina, codeina)', 'Heroina', 'Opiaceos sinteticos (propoxifeno, nailbufina)',
    'Con utilidad medica (Prozac, Paxil, Carbamazepina)'
]
    df['Count_ileg'] = df[list_av_ileg].max(axis=1)
    df['Count_leg'] = (df['Tabaco'] + df['Alcohol']).clip(upper=1)
    df['CasosRiecs'] = np.where(df['Count_ileg'] ==1,1, np.where(df['Count_leg'] ==1,2,0) )
    return df

def Recod_data(df):
    list_aux = ['edocivil', 'escolar', 'ocupa2', 'nivsoc', 'sexo']
    df[list_aux] = (
    df[list_aux]
    .replace(r'^\s*$', np.nan, regex=True)  # ← convierte '' o '   ' en NaN
    .fillna(0)                              # ← rellena NaN con 0
    .astype(int)                            # ← convierte a int
    )
    df2 = pd.DataFrame()
    df2['caso'] = df['caso']
    df2['FolioId'] = df['folio'].astype(str) + '-' + df['cec'].astype(str)
    df2['Edad_Años'] = df['edad']
    df2['Edad'] = df['edad'].apply(rangos_edades)
    df2['Sexo'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer', 9:  np.nan})
    df2['Estado'] = df['Estado']
    df2['CentroCostoId'] = df['CentroCostoId']
    print(f"Total de registros: {len(df2)}")
    df2 = df2[(df2['CentroCostoId'] > 58) |(df2['CentroCostoId'].isin([48, 49]))]
    print (f"Total de registros después de filtrar por CentroCostoId: {len(df2)}")
    df2['Migracion'] = 0
    df2['ComunEstadoCivilId'] = df['edocivil'].map({0: 'Sin Dato', 1: 'Soltero(a)', 2: 'Casado(a)', 3: 'Unión Libre', 4: 'Separado(a)', 5: 'Divorciado(a)', 6: 'Viudo(a)', 9: 'Sin Dato'})
    df2['ComunEscolaridadId'] = df['escolar'].map({0: 'Sin Dato', 10: 'Sin Estudios', 20: 'Sin Estudios', 30: 'Sin Estudios', 31: 'Primaria', 32: 'Sin Estudios', 33: 'Sin Estudios', 40: 'Primaria', 41: 'Secundaria', 42: 'Primaria', 43: 'Primaria', 50: 'Secundaria', 51: 'Preparatoria o Carrera Técnica', 52: 'Secundaria', 53: 'Secundaria', 60: 'Secundaria', 61: 'Preparatoria o Carrera Técnica', 62: 'Secundaria', 63: 'Secundaria', 70: 'Preparatoria o Carrera Técnica', 71: 'Estudios Superiores', 72: 'Preparatoria o Carrera Técnica', 73: 'Preparatoria o Carrera Técnica', 80: 'Estudios Superiores', 81: 'Estudios de Posgrado', 82: 'Estudios Superiores', 83: 'Estudios Superiores'})
    df2['ComunOcupacionId'] = df['ocupa2'].map({0: 'Sin Dato', 1: 'Estudiante', 2: 'Estudiante', 3: 'Con actividad laboral', 4: 'Con actividad laboral', 5: 'Sin ocupación', 6: 'Sin ocupación', 7: 'Hogar', 8: 'Pensionado o jubilado', 9 : 'Sin Dato'})
    df2['ComunEstratoSocialId'] = df['nivsoc'].map({0 : 'Sin Dato', 1: 'Alto', 2: 'Medio Alto', 3: 'Medio Bajo', 4: 'Bajo', 5 : 'Muy Bajo', 9: 'Sin Dato'})
    df2['PerteneceComunidadLGBTTTI'] = 0
    df2['PerteneceComunidadIndigena'] = 0
    df2['PoblacionAfromexicanaAfroamericana'] = 0
    df2['DiscapacidadPerceptual'] = 0
    df2['DrogaImpacto'] = df['drogaim'].replace({0:  np.nan , 1:  'Tabaco', 2:  'Alcohol', 3:  'Marihuana', 4:  'Hachis', 5:  'Cocaína', 6:  'Crack', 7:  'Otras Presentaciones (Basuco o pasta base, cocaina negra)', #1: Np.nan es porque lo registran sin Dato
    8: 'Solventes y removedores', 9: 'Pegamento', 10: 'Esmaltes y pinturas', 11: 'Otros (aire comprimido, gasolinas y combustibles)', 12: 'Anfetaminas', 13: 'Metanfetaminas', 14: 'MDMA(extasis) y metanfetaminas alucinogenas (DMT)',
    15: 'Otros (derivados anfetaminicos)', 16: 'LSD' , 17: 'Plantas alucinogenas y derivados', 18: 'Otras (PCP, ketamina, excepto metanfetamina)', 19: 'Benzodiazepinas',
    20: 'Rohypnol', 21: 'Otras Sustancias', 22: 'Heroina', 23: 'Opiaceos sinteticos (propoxifeno, nailbufina)', 24: 'Opio y opiodes (morfina, codeina)', 25: 'Con utilidad medica (Prozac, Paxil, Carbamazepina)', 26: 'Otras Sustancias',
    27: np.nan , 88: np.nan, 99: np.nan})
    df2['Año'] = df['Año']
    df2['Mes'] = df['Mes']
    df2['Semestre'] = df2['Mes'].apply(lambda x: 1 if x in range(1, 7) else 2)
    df2['Mes'] = df['Año'].astype(str) + '-' + df['Mes'].astype(str).str.zfill(2)
    df2['Semestre'] = df['Año'].astype(str) + '-' + df['Semestre'].astype(str).str.zfill(2)
    df2['ConsumoDeDrogas'] = df['motivo1']
    df2['ConsumoDeBebidasAlcoholicas'] = df['motivo2']
    df2['ConsumoDeTabaco'] = df['motivo3']
    df2['Ludopatia'] = 0
    if 'motivo4' in df.columns:
        df2['Otro'] = df['motivo4']
    elif 'motivo5a' in df.columns:
        df2['Otro'] = df['motivo5a']
    else:
        df2['Otro'] = 0
    df2['TrastornosMentales'] = 0
    df2['Depresion'] = 0
    df2['Psicosis'] = 0
    df2['Epilepsia'] = 0
    df2['Demencia'] = 0
    df2['Autolesion'] = 0
    df2['Suicidio'] = 0
    df2['Ansiedad'] = 0
    df2['ProblemasSalud'] = df['prob1']
    df2['ProblemasFamiliares'] = df['prob3']
    df2['AccidentesAsociados'] = df['prob2']
    df2['ProblemasEscolares'] = df['prob4']
    df2['ProblemasLaborales'] = df['prob5']
    df2['ProblemasPsicologicos'] = df['prob6']
    df2['ProblemasLegales'] = df['prob7']
    df2['ConductaAntisocial'] = df['prob8']
    df2['ProblemasOtros'] = df['prob9']

    df2['Tabaco'] = df['tab1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Alcohol'] = df['alcoav'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Marihuana'] = df['mar1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Hachis'] = df['mar2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Cocaína'] = df['coc1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Crack'] = df['coc2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otras Presentaciones (Basuco o pasta base, cocaina negra)'] = df['coc3av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Solventes y removedores'] = df['inh1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Pegamento'] = df['inh2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Esmaltes y pinturas'] = df['inh3av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otros (aire comprimido, gasolinas y combustibles)'] = df['inh4av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Anfetaminas'] = df['est1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Metanfetaminas'] = df['est2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['MDMA(extasis) y metanfetaminas alucinogenas (DMT)'] = df['est3av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otros (derivados anfetaminicos)'] = df['est4av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['LSD'] = df['alu1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Plantas alucinogenas y derivados'] = df['alu2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otras (PCP, ketamina, excepto metanfetamina)'] = df['alu3av'].replace({1: 1, 2: 0, 9: np.nan}) 
    df2['Benzodiazepinas'] = df['dep1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Rohypnol'] = df['dep2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otras Sustancias'] = df['dep3av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Heroina'] = df['opi1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Opiaceos sinteticos (propoxifeno, nailbufina)'] = df['opi2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Opio y opiodes (morfina, codeina)'] = df['opi3av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Con utilidad medica (Prozac, Paxil, Carbamazepina)'] = df['osu1av'].replace({1: 1, 2: 0, 9: np.nan})
    df2['Otras Sustancias']  = df['osu2av'].replace({1: 1, 2: 0, 9: np.nan})
    df2 = UltimoMes(df, df2)
    df2 = Edad_inicio(df, df2)
    df2 = DataEsp(df2)
    return df2

def main ():
    list_files = ['\\data\\2011A_1', '\\data\\2011B_1', '\\data\\2012A_1', '\\data\\2012B_1', '\\data\\2013A_1', '\\data\\2013B_1']
    for file in list_files:
        filepath = os.getcwd() + file
        df = Read_data(filepath)
        df = Modify_data(df, file)
        df = Recod_data(df)
        dfconcat = pd.concat([dfconcat, df]) if 'dfconcat' in locals() else df
    return dfconcat

In [3]:
df = main()
df.sort_values(by ='Año', inplace=True)
# df = df[df['CasosRiecs'].isin([1,2])]
print (f"Total de registros después de filtrar por CasosRiecs: {len(df)}")
df = df[df['caso'].isin([1,2])]
print (f"Total de registros después de filtrar por caso: {len(df)}")
df.drop(columns=['CasosRiecs', 'Count_ileg', 'Count_leg'], inplace=True)
# df.drop_duplicates(subset=['FolioId'], keep='last', inplace=True)

C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:2: DtypeWarning: Columns (289,290) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{filepath}.csv')


Total de registros: 18307
Total de registros después de filtrar por CentroCostoId: 18251


C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioPegamento'] = df['inh2e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

Total de registros: 15254
Total de registros después de filtrar por CentroCostoId: 15235


C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioPegamento'] = df['inh2e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

...

Total de registros: 18407
Total de registros después de filtrar por CentroCostoId: 18387


C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioPegamento'] = df['inh2e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

Total de registros: 15879
Total de registros después de filtrar por CentroCostoId: 15865


C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioPegamento'] = df['inh2e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2['EdadInicioEsmaltes y pinturas'] = df['inh3e']
C:\Users\franc\AppData\Local\Temp\ipykernel_27108\2390189095.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

Total de registros después de filtrar por CasosRiecs: 103414
Total de registros después de filtrar por caso: 94547


In [4]:
df.to_csv(f'{os.getcwd()}\\result\\db_2011_13.csv', index=False)